In [1]:
import sys
from ortools.sat.python import cp_model

In [ ]:
def read_input():
    data = list(map(int, sys.stdin.read().split()))
    N, M = data[0], data[1]
    idx = 2
    f = [0] * M
    for _ in range(M):
        f[i] = data[idx]; idx += 1
    Q = [0] * M
    for _ in range(M):
        Q[i] = data[idx]; idx += 1
    c = [[0] * N for _ in range(M)]
    for i in range(M):
        for j in range(N):
            c[i][j] = data[idx]; idx += 1

    d = [0] * N
    for i in range(N):
        d[i] = data[idx]; idx += 1
    return N, M, f, Q, c, d

In [ ]:
from collections import defaultdict
def solve_ortools(N, M, f, Q, c, d):
    model = cp_model.CpModel()
    solver = cp_model.CpSolver()

    x = []
    y = [model.new_bool_var(f"y[{i}]") for i in range(M)]
    cost2_elements = []
    for i in range(M):
        x.append([model.new_int_var(0, d[j], f"x[{i}][{j}]") for j in range(N)])

        for j in range(N):
            model.add(x[i][j] <= d[j] * y[i])
        cost2_elements.append(sum(x[i][j] * c[i][j] for j in range(N)))

    for i in range(M):
        model.add(sum(x[i][j] for j in range(N)) <= Q[i] * y[i])

    for j in range(N):
        model.add(sum(x[i][j] for i in range(M)) == d[j])

    cost1 = sum(f[i] * y[i] for i in range(M))
    cost2 = sum(cost2_elements)
    model.minimize(cost1 + cost2)

    solver.parameters.max_time_in_seconds = 5
    status = solver.solve(model)

    plan = defaultdict(list)
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(solver.ObjectiveValue())
        for i in range(M):
            if solver.value(y[i]) == 1:
                plan[i] = [solver.value(x[i][j]) for j in range(N)]

        for key, value in plan.items():
            print(f"Mở điểm dịch vụ {key + 1}")
            print(f"Lượng hàng hóa phục vụ của điểm {key + 1} cho {N} khách hàng là: {value}")
    else:
        print("Không tồn tại phương án mở dịch vụ khả thi!")